In [1]:
import os
import pandas as pd
from google.colab import drive
from torch.cuda import is_available
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report


drive.mount('/content/drive')
#!ls '/content/drive/MyDrive'
#!cp ЛЦТ/* {source_dir}
source_dir = '/content/drive/MyDrive/2025_09_LCT/data/source'
data_dir = dest_dir = '/content/drive/MyDrive/2025_09_LCT/data/auto'
#reviews_df = pd.read_csv(os.path.join(source_dir, 'reviews_banki-ru_25-09.csv'), index_col='id')
#reviews_df.head()
!ls {dest_dir}

Mounted at /content/drive
100ep_full_data_model.pth	      category_classifier_18_classes.cbm
18_cat_classifier_full_data.cbm       category_classifier_30_classes.cbm
251004_cat_classifier_30_classes.cbm  category_classifier.cbm
251004_sent_classifier_3_classes.cbm  joint_classification_model.cbm
80ep_85perc_data_model_fixed.pth      sent_classifier_3_classes.cbm
batch_1f1e4fd8_083151.json	      sentences_1296_processed.csv
batch_371d859b_093151.json	      sentences_1639_processed.csv
batch_4e5cd182_115229.json	      sentences_1962_processed.csv
batch_60e237d8_112621.json	      sentences_2297_processed.csv
batch_65f0ccbd_084912.json	      sentences_2622_processed.csv
batch_91614183_081309.json	      sentences_2969_processed.csv
batch_aab1f84c_092526.json	      sentences_314_processed.csv
batch_e57f8250_081819.json	      sentences_3282_processed.csv
batch_f13a2f63_090317.json	      sentences_3629_processed.csv
batch_f68a2733_114044.json	      sentences_656_processed.csv
batch_f858725c_10

In [2]:
from torch.cuda import is_available

RANDOM_STATE = RANDOM_SEED = 42
device = DEVICE = 'GPU' if is_available() else 'CPU'

#CATEGORIES_D = pd.read_csv(os.path.join(source_dir, 'categories.csv')).set_index('id')['наименование'].to_dict()
CATEGORIES_D = {
    1: "Дебетовые карты",
    2: "Кредитные карты",
    3: "Вклады",
    4: "Накопительные счета",
    5: "Потребительские кредиты",
    6: "Ипотека",
    7: "Страхование",
    8: "Денежные переводы, СБП",
    9: "Интернет-банк, мобильный банк и приложение",
    10: "Премиум-обслуживание",
    11: "Работа колл-центра и клиентского сервиса",
    12: "Безопасность и защита",
    13: "Общее впечатление о банке",
    14: "Кэшбэк, промокоды, бонусы, акции, приведи друга",
    15: "Прочие банковские услуги и сервисы",
    16: "Сравнение с конкурентами",
    17: "Тарифы, комиссии, прозрачность условий",
    18: "Эскалация и угроза жалоб"
}

# Обратный маппинг для поиска id по названию
CATEGORIES_REVERSE = {v: k for k, v in CATEGORIES_D.items()}
NUM_CLASSES = len(CATEGORIES_D)
NUM_SENT_CLASSES = 3
TEST_SIZE = .15


pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

assert NUM_CLASSES==18

In [3]:
import json
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
#from sklearn.multioutput import MultiOutputClassifier
#from catboost import CatBoostClassifier, Pool
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')


In [4]:
#!ls {source_dir}
reviews_ds = pd.read_csv(os.path.join(source_dir, 'reviews_banki-ru_25-09.csv'))

### data load and split

In [34]:
import json
import pandas as pd
from typing import List, Dict

import json

def safe_load_json(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Файл {file_path} не найден")
        return None
    except json.JSONDecodeError as e:
        print(f"Ошибка парсинга JSON: {e}")
        return None
    except Exception as e:
        print(f"Неизвестная ошибка: {e}")
        return None

def load_json_files(names):
    """
    Загружает все JSON файлы из указанной папки
    """
    all_data = []

    for filename in names:
        #if filename.endswith('.json'):
            file_path = os.path.join(source_dir, filename)
            try:
                #with open(file_path, 'r', encoding='utf-8') as f:
                    #data = json.load(f)
                    data = safe_load_json(file_path)
                    all_data.extend(data)
                    #all_data.append(data)
            except Exception as e:
                print(f"Ошибка при загрузке файла {filename}: {e}")

    return all_data
data = load_json_files(['200_labeled_reviews_new_02.10.2025_v3.txt', '600_labeled_reviews_02.10.2025 v4.txt'])
#data = load_json_files(['600_labeled_reviews_02.10.2025 v4.txt'])
len(data)
data[1]


# Список категорий для маппинга

def load_reviews_json(data) -> pd.DataFrame:
    records = []
    for review in data:
        review_id = review['id']

        for idx, sentence in enumerate(review['sentences']):
            # Берем первую категорию как основную (или можно все обработать)
            categories = sentence['category']
            main_category = sentence.get('main_category', categories[0] if categories else 'Другие темы')

            # Маппим категорию в cat_id
            cat_id = CATEGORIES_REVERSE.get(main_category, 0)  # 0 для неизвестных
            if cat_id==0:
                if 'Кэшбэк, промокоды, бонусы, акции' in main_category:
                    cat_id = 14
                if 'Работа колл-центра' in main_category:
                    cat_id = 11
                main_category = CATEGORIES_D.get(cat_id, 0)
                break
                #if cat_id==0: print(sentence.get('main_category'))
            record = {
                'sentence_id': f"{review_id}_{idx:03d}",  # где idx - позиция предложения (0, 1, 2...)
                'review_id': review_id,
                'cat_id': cat_id,
                #'categories': categories,
                'main_category': main_category,
                'sentiment': sentence['sentiment'],
                'sentence_text': sentence['text']
            }
            records.append(record)

    df = pd.DataFrame(records)
    return df

# Использование
tr_val_df = load_reviews_json(data)
print(f"Загружено {len(tr_val_df)} предложений")
print(f"Уникальных отзывов: {tr_val_df['review_id'].nunique()}")
tr_val_df.head()

Загружено 5122 предложений
Уникальных отзывов: 448


,sentence_id,review_id,cat_id,main_category,sentiment,sentence_text
0,11476502_000,11476502,1,Дебетовые карты,0,Оформила дебетовую карту Юнион пей в январе 24г. Стоимость оформления 5000р!.
1,11476502_001,11476502,14,"Кэшбэк, промокоды, бонусы, акции, приведи друга",0,"На тот момент была акции, что вернем 5000р. при условии, что продержите остаток от 50 000,00 на карте или будут траты."
2,11476502_002,11476502,14,"Кэшбэк, промокоды, бонусы, акции, приведи друга",0,"Положить деньги нужно было в этот же день, что собственно говоря и было сделано."
3,11476502_003,11476502,14,"Кэшбэк, промокоды, бонусы, акции, приведи друга",-1,"26.04.24 был последний день начисления денежных средств, которых не начислили."
4,11476502_004,11476502,14,"Кэшбэк, промокоды, бонусы, акции, приведи друга",0,Условия акции Я выполнила.


In [35]:
def split_data_by_review_id(df, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    df = df.copy()

    # Извлекаем review_id из sentence_id (часть до подчеркивания)
    #df['review_id'] = df['sentence_id'].apply(lambda x: x.split('_')[0] if '_' in x else x)

    # Получаем уникальные review_id
    unique_review_ids = df['review_id'].unique()

    # Разделяем уникальные review_id на train/val
    train_review_ids, val_review_ids = train_test_split(
        unique_review_ids,
        test_size=test_size,
        random_state=random_state
    )

    # Создаем маски для разделения исходного DataFrame
    train_mask = df['review_id'].isin(train_review_ids)
    val_mask = df['review_id'].isin(val_review_ids)

    df_train = df[train_mask].copy()
    df_val = df[val_mask].copy()

    # Удаляем временный столбец
    #df_train.drop('review_id', axis=1, inplace=True)
    #df_val.drop('review_id', axis=1, inplace=True)

    print(f"Всего уникальных отзывов: {len(unique_review_ids)}")
    print(f"Train: {len(train_review_ids)} отзывов, {len(df_train)} предложений")
    print(f"Val: {len(val_review_ids)} отзывов, {len(df_val)} предложений")

    return df_train, df_val

tr_val_df['review_id'] = tr_val_df['sentence_id'].apply(lambda x: x.split('_')[0])
train_df, val_df = split_data_by_review_id(tr_val_df)
val_df, test_df = split_data_by_review_id(val_df)

Всего уникальных отзывов: 448
Train: 380 отзывов, 4252 предложений
Val: 68 отзывов, 870 предложений
Всего уникальных отзывов: 68
Train: 57 отзывов, 732 предложений
Val: 11 отзывов, 138 предложений


### get review targets



In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from typing import List, Dict, Optional
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


In [64]:
# !!! вот это я упустил

print(train_df['sentiment'].unique())
def map_sent(df):
    df.loc[df['sentiment']>0, 'sentiment']=3
    df.loc[df['sentiment']==0, 'sentiment']=2
    df.loc[df['sentiment']<0, 'sentiment']=1
    '''train_df.loc[train_df['sentiment']>0, 'sentiment']=3
    val_df.loc[val_df['sentiment']>0, 'sentiment']=3
    train_df.loc[train_df['sentiment']==0, 'sentiment']=2
    val_df.loc[val_df['sentiment']==0, 'sentiment']=2
    train_df.loc[train_df['sentiment']<0, 'sentiment']=1
    val_df.loc[val_df['sentiment']<0, 'sentiment']=1'''
map_sent(train_df)
map_sent(val_df)
map_sent(test_df)

'''def map_sent_tmp():
    train_df.loc[train_df['sentiment']>0, 'sentiment']=1
    val_df.loc[val_df['sentiment']>0, 'sentiment']=1
    train_df.loc[train_df['sentiment']==0, 'sentiment']=2
    val_df.loc[val_df['sentiment']==0, 'sentiment']=2
    train_df.loc[train_df['sentiment']<0, 'sentiment']=3
    val_df.loc[val_df['sentiment']<0, 'sentiment']=3

map_sent_tmp()'''
print(train_df['sentiment'].unique())

[1 2 3]
[1 2 3]


In [37]:
def aggregate_sentiments_by_category(unique_pairs):
    """Агрегирует сентименты по категориям (выбирает наиболее частый для каждой категории)"""

    # Группируем пары по категориям
    from collections import defaultdict
    category_sentiments = defaultdict(list)

    for cat_id, sentiment in unique_pairs:
        category_sentiments[cat_id].append(sentiment)

    # Для каждой категории выбираем наиболее частый сентимент
    category_sentiment_map = {}
    for cat_id, sentiments in category_sentiments.items():
        # Выбираем моду (наиболее частый сентимент)
        from collections import Counter
        sentiment_counts = Counter(sentiments)
        most_common_sentiment = sentiment_counts.most_common(1)[0][0]
        category_sentiment_map[cat_id] = most_common_sentiment

    return category_sentiment_map

# Исправленная часть в aggregate_review_targets:
def aggregate_review_targets(df):
    """Агрегирует таргет-метки из предложений в отзывы (категории + сентимент)"""

    df = df.copy()
    df['review_id'] = df['sentence_id'].apply(lambda x: x.split('_')[0])

    review_targets = {}

    for review_id, group in df.groupby('review_id'):
        # Собираем уникальные пары (cat_id, sentiment) для каждого отзыва
        unique_pairs = group[['cat_id', 'sentiment']].drop_duplicates().values.tolist()

        # Исправленный подсчет сентиментов
        cat_ids = list(set([pair[0] for pair in unique_pairs]))
        category_sentiment_map = aggregate_sentiments_by_category(unique_pairs)

        review_targets[review_id] = {
            'true_cat_ids': cat_ids,
            'true_sentiments': list(set([pair[1] for pair in unique_pairs])),  # все уникальные сентименты
            'true_category_sentiment_map': category_sentiment_map,  # сентимент для каждой категории
            'true_cat_sentiment_pairs': unique_pairs
        }

    # Создаем DataFrame с результатами
    result_df = pd.DataFrame([
        {
            'review_id': review_id,
            'true_cat_ids': targets['true_cat_ids'],
            'true_sentiments': targets['true_sentiments'],
            'true_category_sentiment_map': targets['true_category_sentiment_map'],
            'true_cat_sentiment_pairs': targets['true_cat_sentiment_pairs']
        }
        for review_id, targets in review_targets.items()
    ])

    return result_df

train_review_targets = aggregate_review_targets(train_df)
val_review_targets = aggregate_review_targets(val_df)

print(f"Train reviews: {len(train_review_targets)}")
print(f"Val reviews: {len(val_review_targets)}")
print("\nПример агрегированных таргетов:")
print(train_review_targets.head())

Train reviews: 380
Val reviews: 57

Пример агрегированных таргетов:
  review_id           true_cat_ids true_sentiments  \
0  11324490                [11, 5]          [1, 2]   
1  11336318  [1, 8, 9, 11, 13, 17]       [1, 2, 3]   
2  11336622            [9, 11, 17]          [1, 2]   
3  11344989         [3, 1, 11, 13]             [3]   
4  11345056               [11, 13]             [3]   

               true_category_sentiment_map  \
0                            {5: 2, 11: 1}   
1  {1: 2, 8: 3, 9: 1, 17: 1, 11: 1, 13: 2}   
2                     {9: 2, 11: 1, 17: 1}   
3               {11: 3, 3: 3, 1: 3, 13: 3}   
4                           {11: 3, 13: 3}   

                              true_cat_sentiment_pairs  
0                            [[5, 2], [5, 1], [11, 1]]  
1  [[1, 2], [8, 3], [9, 1], [17, 1], [11, 1], [13, 2]]  
2                           [[9, 2], [11, 1], [17, 1]]  
3                   [[11, 3], [3, 3], [1, 3], [13, 3]]  
4                                   [[11, 3],

In [10]:
#train_df[train_df['sentence_id'].str.contains('11314176')]
#reviews_ds[reviews_ds['id']==11314037]

In [38]:
!pip install catboost -qq
from catboost import CatBoostClassifier, Pool

cat_f_path = os.path.join(dest_dir, '18_cat_classifier_full_data.cbm')
sent_f_path = os.path.join(dest_dir, 'sent_classifier_3_classes.cbm')
cat_classifier = CatBoostClassifier()
cat_classifier.load_model(cat_f_path)
sent_classifier = CatBoostClassifier()
sent_classifier.load_model(sent_f_path)

In [48]:
#all_probs = cat_classifier.predict_proba(val_df['sentence_text'].tolist())
def add_probs(df, ):
    pool = Pool(df['sentence_text'].values, text_features=[0])
    cat_probs = cat_classifier.predict_proba(pool)
    print('cat_probs', cat_probs.shape[1])
    cat_prob_columns = [f'cat_prob_cat_{i+1}' for i in range(cat_probs.shape[1])]
    df[cat_prob_columns] = cat_probs
    sent_probs = sent_classifier.predict_proba(pool)
    #print('sent_probs', sent_probs.shape[1])
    sent_prob_columns = [f'sent_prob_cat_{i+1}' for i in range(sent_probs.shape[1])]
    df[sent_prob_columns] = sent_probs

'''
val_pool = Pool(val_df['sentence_text'].values, text_features=[0])
val_probs = cat_classifier.predict_proba(val_pool)
print(train_probs.shape, val_probs.shape)
all_probs[:1]'''
add_probs(train_df)
add_probs(val_df)
add_probs(test_df)
val_df.columns

cat_probs 18
cat_probs 18
cat_probs 18


Index(['sentence_id', 'review_id', 'cat_id', 'main_category', 'sentiment',
       'sentence_text', 'cat_prob_cat_1', 'cat_prob_cat_2', 'cat_prob_cat_3',
       'cat_prob_cat_4', 'cat_prob_cat_5', 'cat_prob_cat_6', 'cat_prob_cat_7',
       'cat_prob_cat_8', 'cat_prob_cat_9', 'cat_prob_cat_10',
       'cat_prob_cat_11', 'cat_prob_cat_12', 'cat_prob_cat_13',
       'cat_prob_cat_14', 'cat_prob_cat_15', 'cat_prob_cat_16',
       'cat_prob_cat_17', 'cat_prob_cat_18', 'sent_prob_cat_1',
       'sent_prob_cat_2', 'sent_prob_cat_3'],
      dtype='object')

In [13]:
#train_df['review_id'] = train_df['sentence_id'].apply(lambda x: x.split('_')[0])
#val_df['review_id'] = val_df['sentence_id'].apply(lambda x: x.split('_')[0])


In [49]:
class ReviewDataset(Dataset):
    def __init__(self, sentences_df, targets_df=None, num_categories=30, num_sentiments=3, max_seq_len=50):
        self.sentences_df = sentences_df
        self.targets_df = targets_df.set_index('review_id') if targets_df is not None else None
        self.num_categories = num_categories
        self.num_sentiments = num_sentiments
        self.max_seq_len = max_seq_len
        self.is_test = targets_df is None

        # Группируем предложения по отзывам
        self.review_groups = sentences_df.groupby('review_id')
        self.review_ids = list(self.review_groups.groups.keys())

    def __len__(self):
        return len(self.review_ids)

    def __getitem__(self, idx):
        review_id = self.review_ids[idx]
        group = self.review_groups.get_group(review_id).sort_values('sentence_id')

        # Берем вероятности категорий и сентиментов
        cat_prob_cols = [f'cat_prob_cat_{i+1}' for i in range(self.num_categories)]
        sent_prob_cols = [f'sent_prob_cat_{i+1}' for i in range(self.num_sentiments)]

        cat_probs = group[cat_prob_cols].values.astype(np.float32)
        sent_probs = group[sent_prob_cols].values.astype(np.float32)

        # Объединяем фичи
        sentence_features = np.concatenate([cat_probs, sent_probs], axis=1)

        # Паддинг
        if len(sentence_features) > self.max_seq_len:
            sentence_features = sentence_features[:self.max_seq_len]
        else:
            padding = np.zeros((self.max_seq_len - len(sentence_features),
                              self.num_categories + self.num_sentiments), dtype=np.float32)
            sentence_features = np.vstack([sentence_features, padding])

        # ТЕСТОВЫЙ РЕЖИМ - без таргетов
        if self.is_test:
            return torch.FloatTensor(sentence_features), review_id

        # ТРЕНИРОВОЧНЫЙ РЕЖИМ - с таргетами
        true_cat_ids = self.targets_df.loc[review_id, 'true_cat_ids']
        category_sentiment_map = self.targets_df.loc[review_id, 'true_category_sentiment_map']

        # Собираем все пары (категория, сентимент) из отзыва
        all_pairs = []
        for _, row in group.iterrows():
            cat_id = row['cat_id']
            sentiment = row['sentiment']
            all_pairs.append((cat_id, sentiment))

        # Агрегируем по категориям
        from collections import defaultdict, Counter
        cat_sentiments = defaultdict(list)
        for cat_id, sentiment in all_pairs:
            cat_sentiments[cat_id].append(sentiment)

        # ПРАВИЛЬНАЯ ЛОГИКА ДЛЯ СЕНТИМЕНТОВ
        final_category_sentiment = {}
        for cat_id, sentiments in cat_sentiments.items():
            unique_sents = set(sentiments)

            # Есть негативный (1) ИЛИ позитивный (3) сентимент
            if 1 in unique_sents or 3 in unique_sents:
                # Удаляем нейтральные (2) из рассмотрения
                filtered_sents = [s for s in sentiments if s != 2]

                if not filtered_sents:  # если после фильтрации ничего не осталось
                    final_sentiment = 2
                # Если остались и негатив и позитив - ставим нейтрал (2)
                elif 1 in filtered_sents and 3 in filtered_sents:
                    final_sentiment = 2
                else:
                    # Берем преобладающий крайний сентимент
                    sentiment_counts = Counter(filtered_sents)
                    final_sentiment = sentiment_counts.most_common(1)[0][0]
            else:
                # Только нейтральные упоминания
                final_sentiment = 2 if sentiments else 0

            final_category_sentiment[cat_id] = final_sentiment

        # One-hot для категорий
        cat_target = np.zeros(self.num_categories, dtype=np.float32)
        for cat_id in final_category_sentiment.keys():
            if 1 <= cat_id <= self.num_categories and final_category_sentiment[cat_id] != 0:
                cat_target[cat_id - 1] = 1

        # One-hot для сентиментов по категориям
        sentiment_target = np.zeros((self.num_categories, self.num_sentiments), dtype=np.float32)
        for cat_id, sentiment in final_category_sentiment.items():
            if 1 <= cat_id <= self.num_categories and 1 <= sentiment <= self.num_sentiments:
                sentiment_target[cat_id - 1, sentiment - 1] = 1

        return (torch.FloatTensor(sentence_features),
                torch.FloatTensor(cat_target),
                torch.FloatTensor(sentiment_target))

MAX_SEQ_LEN =50
'''
# Создаем датасеты
train_dataset = ReviewDataset(train_df, train_review_targets,
                             num_categories=NUM_CLASSES, num_sentiments=NUM_SENT_CLASSES, max_seq_len=MAX_SEQ_LEN)
val_dataset = ReviewDataset(val_df, val_review_targets,
                           num_categories=NUM_CLASSES, num_sentiments=NUM_SENT_CLASSES, max_seq_len=MAX_SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

for batch_x, batch_y_cat, batch_y_sent in train_loader:
    break
batch_x.shape, batch_y_cat.shape, batch_y_sent.shape'''
None

In [17]:
'''#11548205
#12319816 сравнение с конкурентами
# 11504855 - положительный
#tmp_df = val_df[val_df['review_id']=='11504855'].sort_values('sentence_id')
tmp_df = val_df[val_df['review_id'].isin(['11504855', '12319816'])].sort_values('sentence_id')
display(tmp_df.sample(3))
tmp_dataset = ReviewDataset(tmp_df, val_review_targets,
                             num_categories=NUM_CLASSES, num_sentiments=NUM_SENT_CLASSES, max_seq_len=MAX_SEQ_LEN)
tmp_loader = DataLoader(tmp_dataset, batch_size=32, shuffle=False)

tmp_dataset.__getitem__(1)'''

"#11548205\n#12319816 сравнение с конкурентами\n# 11504855 - положительный\n#tmp_df = val_df[val_df['review_id']=='11504855'].sort_values('sentence_id')\ntmp_df = val_df[val_df['review_id'].isin(['11504855', '12319816'])].sort_values('sentence_id')\ndisplay(tmp_df.sample(3))\ntmp_dataset = ReviewDataset(tmp_df, val_review_targets,\n                             num_categories=NUM_CLASSES, num_sentiments=NUM_SENT_CLASSES, max_seq_len=MAX_SEQ_LEN)\ntmp_loader = DataLoader(tmp_dataset, batch_size=32, shuffle=False)\n\ntmp_dataset.__getitem__(1)"

In [ ]:
NUM_SENT_CLASSES, CATEGORIES_D

In [50]:
train_dataset = ReviewDataset(train_df, train_review_targets,
                             num_categories=NUM_CLASSES, num_sentiments=NUM_SENT_CLASSES, max_seq_len=MAX_SEQ_LEN)
val_dataset = ReviewDataset(val_df, val_review_targets,
                           num_categories=NUM_CLASSES, num_sentiments=NUM_SENT_CLASSES, max_seq_len=MAX_SEQ_LEN)
test_dataset = ReviewDataset(test_df, targets_df=None,
                           num_categories=NUM_CLASSES, num_sentiments=NUM_SENT_CLASSES, max_seq_len=MAX_SEQ_LEN)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)


for batch_x, batch_y_cat, batch_y_sent in train_loader:
    break
len(train_dataset), len(val_dataset), len(test_dataset), batch_x.shape, batch_y_cat.shape, batch_y_sent.shape

(380,
 57,
 11,
 torch.Size([32, 50, 21]),
 torch.Size([32, 18]),
 torch.Size([32, 18, 3]))

In [42]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

class ReviewCNN1D(nn.Module):
    def __init__(self,
                 num_categories=NUM_CLASSES,
                 num_sentiments=NUM_SENT_CLASSES,
                 hidden_dim=128,
                 num_filters=64,
                 kernel_size=3,
                 dropout=0.3):

        super().__init__()

        self.num_categories = num_categories
        self.num_sentiments = num_sentiments

        # Input: [seq_len, num_categories + num_sentiments] = [seq_len, 33]
        input_dim = num_categories + num_sentiments  # 30 + 3 = 33

        self.conv1d = nn.Sequential(
            nn.Conv1d(input_dim, num_filters, kernel_size, padding=kernel_size//2),  # [33, 64, 3]
            nn.BatchNorm1d(num_filters),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.global_max_pool = nn.AdaptiveMaxPool1d(1)

        combined_features = num_filters * 2

        self.category_head = nn.Sequential(
            nn.Linear(combined_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_categories),
            nn.Sigmoid()
        )
        #print('num_sentiments', num_sentiments)
        self.sentiment_head = nn.Sequential(
            nn.Linear(combined_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_categories * num_sentiments)
            #nn.Linear(hidden_dim, num_sentiments)
        )

    def forward(self, x):
        # x: [batch_size, seq_len, 33] -> [batch_size, 33, seq_len]
        x = x.transpose(1, 2)  # [32, 33, 50]
        x = self.conv1d(x)  # [32, 64, 50]
        avg_pool = self.global_avg_pool(x).squeeze(-1)  # [32, 64]
        max_pool = self.global_max_pool(x).squeeze(-1)  # [32, 64]
        combined = torch.cat([avg_pool, max_pool], dim=1)  # [32, 128]

        category_probs = self.category_head(combined)  # [32, NUM_CLASSES]
        #print('category_probs.shape', category_probs.shape)
        sentiment_logits = self.sentiment_head(combined)
        #print('sentiment_logits.shape', sentiment_logits.shape)
        #print('self.num_categories, self.num_sentiments',self.num_categories, self.num_sentiments)
        #print(sentiment_logits.view(-1, self.num_categories, self.num_sentiments))
        sentiment_probs = F.softmax(
            sentiment_logits.view(-1, self.num_categories, self.num_sentiments),
            dim=-1
        )
        #sentiment_probs = F.softmax(sentiment_logits, dim=-1)  # [batch_size, 3]

        return category_probs, sentiment_probs
'''
model = ReviewCNN1D(
    num_categories=NUM_CLASSES,
    num_sentiments=NUM_SENT_CLASSES,
)
train_with_validation(model, train_loader, val_loader, num_epochs=10, lr=0.001, patience=15)'''
None

In [43]:
def train_with_validation(model, train_loader, val_loader, num_epochs=10, lr=0.001, patience=3):
    criterion_cat = nn.BCELoss()
    criterion_sent = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_f1_cat = 0
    patience_counter = 0

    for epoch in range(num_epochs):
        # Обучение
        model.train()
        train_loss = 0.0
        for batch_x, batch_y_cat, batch_y_sent in train_loader:
            optimizer.zero_grad()
            cat_probs, sent_probs = model(batch_x)

            # Маска для существующих категорий с сентиментом
            mask = (batch_y_sent.sum(dim=2) > 0).view(-1)

            cat_loss = criterion_cat(cat_probs, batch_y_cat)

            if mask.sum() > 0:
                sent_loss = criterion_sent(
                    sent_probs.view(-1, NUM_SENT_CLASSES)[mask],
                    batch_y_sent.view(-1, NUM_SENT_CLASSES).argmax(dim=1)[mask]
                )
            else:
                sent_loss = 0.0

            total_loss = cat_loss + sent_loss
            total_loss.backward()
            optimizer.step()
            train_loss += total_loss.item()

        # Валидация с раздельными метриками
        val_loss, val_f1_cat, val_f1_sent = validate_with_metrics(model, val_loader, criterion_cat, criterion_sent)

        print(f'Epoch {epoch+1}/{num_epochs}:')
        print(f'  Train Loss: {train_loss/len(train_loader):.4f}')
        print(f'  Val Loss: {val_loss:.4f}')
        print(f'  Cat F1: {val_f1_cat:.4f}, Sent F1: {val_f1_sent:.4f}')

        # Early stopping по F1 категорий (или можно по взвешенной сумме)
        if val_f1_cat > best_f1_cat:
            best_f1_cat = val_f1_cat
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pth')
            print('  ↳ New best model saved!')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'  ↳ Early stopping after {epoch+1} epochs')
                break

    model.load_state_dict(torch.load('best_model.pth'))
    return model

def validate_with_metrics(model, val_loader, criterion_cat, criterion_sent, threshold=0.3):
    model.eval()
    val_loss = 0.0
    all_cat_targets = []
    all_cat_predictions = []
    all_sent_targets = []
    all_sent_predictions = []

    with torch.no_grad():
        for batch_x, batch_y_cat, batch_y_sent in val_loader:
            cat_probs, sent_probs = model(batch_x)

            # Маска для существующих категорий с сентиментом
            mask = (batch_y_sent.sum(dim=2) > 0).view(-1)

            cat_loss = criterion_cat(cat_probs, batch_y_cat)

            if mask.sum() > 0:
                sent_loss = criterion_sent(
                    sent_probs.view(-1, NUM_SENT_CLASSES)[mask],
                    batch_y_sent.view(-1, NUM_SENT_CLASSES).argmax(dim=1)[mask]
                )
            else:
                sent_loss = 0.0

            total_loss = cat_loss + sent_loss
            val_loss += total_loss.item()

            # Для метрик категорий
            all_cat_targets.append(batch_y_cat.numpy())
            all_cat_predictions.append((cat_probs.numpy() > threshold).astype(int))

            # Для метрик сентиментов (только для существующих категорий)
            if mask.sum() > 0:
                all_sent_targets.append(batch_y_sent.view(-1, NUM_SENT_CLASSES).argmax(dim=1)[mask].numpy())
                all_sent_predictions.append(sent_probs.view(-1, NUM_SENT_CLASSES).argmax(dim=1)[mask].numpy())

    # Вычисляем F1 для категорий
    from sklearn.metrics import f1_score
    all_cat_targets = np.vstack(all_cat_targets)
    all_cat_predictions = np.vstack(all_cat_predictions)
    f1_cat = f1_score(all_cat_targets, all_cat_predictions, average='micro', zero_division=0)

    # Вычисляем F1 для сентиментов
    if all_sent_targets:
        all_sent_targets = np.hstack(all_sent_targets)
        all_sent_predictions = np.hstack(all_sent_predictions)
        f1_sent = f1_score(all_sent_targets, all_sent_predictions, average='micro', zero_division=0)
    else:
        f1_sent = 0.0

    avg_val_loss = val_loss / len(val_loader)
    return avg_val_loss, f1_cat, f1_sent

In [44]:
model = ReviewCNN1D(
    num_categories=NUM_CLASSES,
    num_sentiments=NUM_SENT_CLASSES,
)
model = train_with_validation(model, train_loader, val_loader, num_epochs=100, lr=0.001, patience=15)

Epoch 1/100:
  Train Loss: 1.4714
  Val Loss: 1.6726
  Cat F1: 0.3263, Sent F1: 0.7200
  ↳ New best model saved!
Epoch 2/100:
  Train Loss: 1.1921
  Val Loss: 1.5004
  Cat F1: 0.3389, Sent F1: 0.7600
  ↳ New best model saved!
Epoch 3/100:
  Train Loss: 1.1293
  Val Loss: 1.3259
  Cat F1: 0.3894, Sent F1: 0.7850
  ↳ New best model saved!
Epoch 4/100:
  Train Loss: 1.0979
  Val Loss: 1.1781
  Cat F1: 0.5252, Sent F1: 0.8250
  ↳ New best model saved!
Epoch 5/100:
  Train Loss: 1.0635
  Val Loss: 1.1018
  Cat F1: 0.5932, Sent F1: 0.8300
  ↳ New best model saved!
Epoch 6/100:
  Train Loss: 1.0280
  Val Loss: 1.0604
  Cat F1: 0.6178, Sent F1: 0.8300
  ↳ New best model saved!
Epoch 7/100:
  Train Loss: 1.0063
  Val Loss: 1.0422
  Cat F1: 0.6525, Sent F1: 0.8400
  ↳ New best model saved!
Epoch 8/100:
  Train Loss: 0.9779
  Val Loss: 1.0191
  Cat F1: 0.6667, Sent F1: 0.8500
  ↳ New best model saved!
Epoch 9/100:
  Train Loss: 0.9625
  Val Loss: 1.0039
  Cat F1: 0.6872, Sent F1: 0.8500
  ↳ New b

In [45]:
#'best_model.pth'
model_name = '86ep_85perc_data_model_fixed_50_50.pth'
torch.save(model.state_dict(), os.path.join(dest_dir, model_name))


In [66]:
model.eval()

# 2. Тест на одном батче
with torch.no_grad():
    #for batch_x, batch_y_cat, batch_y_sent in tmp_loader:
    for batch_x, review_idx in test_loader:
        cat_probs, sent_probs = model(batch_x)
        break
cat_preds = (cat_probs > 0.35).int()
sent_preds = sent_probs.argmax(dim=2)+1

cat_probs[0]

tensor([9.6569e-01, 2.6535e-02, 5.5567e-02, 8.2750e-04, 2.7309e-02, 6.8488e-04,
        3.6254e-04, 3.8274e-03, 9.0598e-03, 4.1360e-03, 9.9787e-01, 1.0336e-02,
        3.1793e-02, 8.5113e-03, 4.5172e-03, 1.8472e-03, 1.9210e-02, 1.4990e-03])

In [83]:
check_idx = 2
test_df[test_df['review_id']==review_idx[check_idx]]

,sentence_id,review_id,cat_id,main_category,sentiment,sentence_text,cat_prob_cat_1,cat_prob_cat_2,cat_prob_cat_3,cat_prob_cat_4,cat_prob_cat_5,cat_prob_cat_6,cat_prob_cat_7,cat_prob_cat_8,cat_prob_cat_9,cat_prob_cat_10,cat_prob_cat_11,cat_prob_cat_12,cat_prob_cat_13,cat_prob_cat_14,cat_prob_cat_15,cat_prob_cat_16,cat_prob_cat_17,cat_prob_cat_18,sent_prob_cat_1,sent_prob_cat_2,sent_prob_cat_3
1040,11684212_000,11684212,12,Безопасность и защита,1,"06.08.2024 в моб. приложении Газпробанка делала перевод по СБП с дебетовой карты другому человеку на Т-Банк, тут же карту заблокировали ссылась на ограничении в лимитах и прислали на мобильный номер след. смс: ""Из-за подозрительной операции 100000.00 RUB в СБП в 13:20 карта ограничена.""",0.013633,0.005855,0.008739,0.012160,0.009386,0.000374,0.007303,0.042597,0.016091,0.007151,0.021857,0.830247,0.003680,0.003684,0.001151,0.003657,0.009724,0.002711,0.812484,0.164093,0.023424
1041,11684212_001,11684212,12,Безопасность и защита,1,"Если это ваша операция, для снятия ограничений направьте это СМС на номер +7903 *** ** 22"", сделала все как указано в СМС от банка и выслала это СМС на указанный номер, так только вот картой от этого я не могу пользоваться, ни себе ни кому то другому деньги перевести я не могу, а перевод необходимо сделать срочно.",0.011283,0.007829,0.007347,0.013611,0.010653,0.000405,0.008936,0.040217,0.019550,0.008911,0.030070,0.810564,0.005811,0.004171,0.001143,0.004939,0.011560,0.003000,0.960557,0.028619,0.010824
1042,11684212_002,11684212,11,Работа колл-центра и клиентского сервиса,1,"Написала в ЧАТ приложения - никто не отвечает, звоню на горячую линию, ожидание от 30 минут, а деньги перевести мне необходимо срочно, и вот уже 30 минут я ожидаю своей очереди в контакт-центре, а перевод повторюсь срочный, не могу перести свои же деньги даже себе на другой - НОРМАЛЬНЫЙ банк, ГАЗПРОМБАНК вы просто пробили ДНО.",0.026232,0.009907,0.017903,0.015754,0.007351,0.000472,0.004163,0.039365,0.036535,0.018031,0.722338,0.019505,0.006483,0.032647,0.001728,0.003079,0.031373,0.007135,0.956877,0.035140,0.007984
1043,11684212_003,11684212,12,Безопасность и защита,1,"Решите мою проблему, разюдлкируйте мою карту что бы я могла закрыть все свои счета, так как вы подводите в самый неподходящий для этого момент.",0.033880,0.009332,0.007492,0.018692,0.009895,0.000344,0.009060,0.024167,0.015886,0.016648,0.047174,0.766701,0.006860,0.006394,0.000965,0.006597,0.017309,0.002604,0.909576,0.074678,0.015745
1044,11684212_004,11684212,13,Общее впечатление о банке,1,НЕ РЕКОМЕНДУЮ никому,0.011549,0.005207,0.006144,0.007081,0.003930,0.001035,0.003259,0.005004,0.005892,0.014334,0.073614,0.006047,0.810350,0.015776,0.004947,0.014149,0.007363,0.004317,0.908134,0.066015,0.025851
2360,11684212_000,11684212,12,Безопасность и защита,1,"06.08.2024 в моб. приложении Газпробанка делала перевод по СБП с дебетовой карты другому человеку на Т-Банк, тут же карту заблокировали ссылась на ограничении в лимитах и прислали на мобильный номер след. смс: ""Из-за подозрительной операции 100000.00 RUB в СБП в 13:20 карта ограничена.",0.013633,0.005855,0.008739,0.012160,0.009386,0.000374,0.007303,0.042597,0.016091,0.007151,0.021857,0.830247,0.003680,0.003684,0.001151,0.003657,0.009724,0.002711,0.812484,0.164093,0.023424
2361,11684212_001,11684212,12,Безопасность и защита,1,"Если это ваша операция, для снятия ограничений направьте это СМС на номер +7903 *** ** 22"", сделала все как указано в СМС от банка и выслала это СМС на указанный номер, так только вот картой от этого я пользоваться не могу, ни себе ни кому то другому деньги перевести я не могу, а перевод необходимо сделать срочно.",0.010760,0.007921,0.007427,0.013394,0.010640,0.000408,0.009177,0.038618,0.020122,0.008604,0.030223,0.812085,0.005965,0.004159,0.001151,0.005019,0.011483,0.002845,0.960557,0.028619,0.010824
2362,11684212_002,11684212,11,Работа колл-центра и клиентского сервиса,1,"Написала в ЧАТ приложения - никто не отвечает, звоню на горячую линию, ожидание от 30 мин

In [84]:
cat_checked_indexes = [idx for idx, k in enumerate(cat_preds[check_idx]) if k.item()==1]
[CATEGORIES_D[idx+1] for idx in cat_checked_indexes]

['Работа колл-центра и клиентского сервиса',
 'Безопасность и защита',
 'Общее впечатление о банке']

In [85]:
sent_preds[check_idx][cat_checked_indexes], sent_preds[check_idx]

(tensor([1, 1, 1]),
 tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))

In [ ]:
def analyze_class_balance(dataset):
    """Анализирует баланс классов в датасете"""

    if dataset.is_test:
        print("Тестовый датасет - нет таргетов для анализа")
        return

    cat_counts = np.zeros(dataset.num_categories)
    sent_counts = np.zeros((dataset.num_categories, dataset.num_sentiments))

    for i in range(len(dataset)):
        _, cat_target, sent_target = dataset[i]
        cat_counts += cat_target.numpy()
        sent_counts += sent_target.numpy()

    print("=== БАЛАНС КАТЕГОРИЙ ===")
    for cat_idx in range(dataset.num_categories):
        count = cat_counts[cat_idx]
        if count > 0:
            print(f"Категория {cat_idx+1}: {int(count)} отзывов ({count/len(dataset)*100:.1f}%)")

    print("\n=== БАЛАНС СЕНТИМЕНТОВ ПО КАТЕГОРИЯМ ===")
    for cat_idx in range(dataset.num_categories):
        total = cat_counts[cat_idx]
        if total > 0:
            sentiments = sent_counts[cat_idx]
            print(f"Категория {cat_idx+1}: негатив={int(sentiments[0])}, нейтраль={int(sentiments[1])}, позитив={int(sentiments[2])}")

# Использование
analyze_class_balance(train_dataset)
analyze_class_balance(val_dataset)

=== БАЛАНС КАТЕГОРИЙ ===
Категория 1: 136 отзывов (35.8%)
Категория 2: 51 отзывов (13.4%)
Категория 3: 25 отзывов (6.6%)
Категория 4: 31 отзывов (8.2%)
Категория 5: 9 отзывов (2.4%)
Категория 6: 3 отзывов (0.8%)
Категория 7: 9 отзывов (2.4%)
Категория 8: 28 отзывов (7.4%)
Категория 9: 86 отзывов (22.6%)
Категория 10: 29 отзывов (7.6%)
Категория 11: 314 отзывов (82.6%)
Категория 12: 44 отзывов (11.6%)
Категория 13: 182 отзывов (47.9%)
Категория 14: 96 отзывов (25.3%)
Категория 15: 6 отзывов (1.6%)
Категория 16: 23 отзывов (6.1%)
Категория 17: 92 отзывов (24.2%)
Категория 18: 11 отзывов (2.9%)

=== БАЛАНС СЕНТИМЕНТОВ ПО КАТЕГОРИЯМ ===
Категория 1: негатив=67, нейтраль=42, позитив=27
Категория 2: негатив=32, нейтраль=14, позитив=5
Категория 3: негатив=12, нейтраль=8, позитив=5
Категория 4: негатив=20, нейтраль=6, позитив=5
Категория 5: негатив=5, нейтраль=2, позитив=2
Категория 6: негатив=2, нейтраль=1, позитив=0
Категория 7: негатив=7, нейтраль=1, позитив=1
Категория 8: негатив=22, нейтр

In [ ]:
sent_preds.shape

torch.Size([5, 18])

In [ ]:
val_df

,sentence_id,cat_id,main_category,sentiment,sentence_text,cat_prob_cat_1,cat_prob_cat_2,cat_prob_cat_3,cat_prob_cat_4,cat_prob_cat_5,cat_prob_cat_6,cat_prob_cat_7,cat_prob_cat_8,cat_prob_cat_9,cat_prob_cat_10,cat_prob_cat_11,cat_prob_cat_12,cat_prob_cat_13,cat_prob_cat_14,cat_prob_cat_15,cat_prob_cat_16,cat_prob_cat_17,cat_prob_cat_18,sent_prob_cat_1,sent_prob_cat_2,sent_prob_cat_3,review_id
590,11724586_000,15,Прочие банковские услуги и сервисы,0,Добрый день!,0.014080,0.007389,0.007192,0.004687,0.003858,0.001158,0.001951,0.005379,0.006735,0.010143,0.051445,0.003181,0.841758,0.014771,0.008894,0.007073,0.008122,0.002185,0.108086,0.849430,0.042484,11724586
591,11724586_001,14,"Кэшбэк, промокоды, бонусы, акции, приведи друга",-1,Произошло некорректное начисление кешбэка за Август 2024.,0.052665,0.022285,0.013664,0.028686,0.006683,0.003214,0.002292,0.016882,0.012672,0.031159,0.033168,0.004669,0.024374,0.646596,0.002369,0.006213,0.090458,0.001953,0.511252,0.280231,0.208517,11724586
592,11724586_002,14,"Кэшбэк, промокоды, бонусы, акции, приведи друга",-1,"Оставлял несколько заявок на горячей линии и мне поступил ответ, что ""ошибок не обнаружено"", и якобы ""не выполнены условия трат на сумму 5000р за отчетный период"".",0.032178,0.004012,0.001782,0.007424,0.002294,0.000132,0.001032,0.002035,0.003450,0.006745,0.062966,0.001535,0.005015,0.841312,0.000600,0.002815,0.023924,0.000747,0.974482,0.019523,0.005995,11724586
593,11724586_003,14,"Кэшбэк, промокоды, бонусы, акции, приведи друга",-1,"Я не согласен с этим решением, так как общая сумма трат за август была более 21784,53 Р.",0.031400,0.019894,0.003669,0.015424,0.004452,0.000547,0.002441,0.006099,0.007942,0.008767,0.023726,0.002883,0.010628,0.778396,0.001116,0.013105,0.068428,0.001083,0.967032,0.026598,0.006370,11724586
985,12319816_000,14,"Кэшбэк, промокоды, бонусы, акции, приведи друга",-1,"Получила 20.05.25 дебетовую карту газпромбанка в офисе,т.к. банк проводит акцию по выплате кэшбэка в размере 35% за коммунальные услуги.Работники банка меня уверили,что с 01.03.25г. остаток на карте должен был оставаться 10 000,0 руб., а не 30 000 руб. В итоге кэшбэка нет, еще и сняли 399 руб. за неизвестные привилегии.",0.058922,0.004014,0.002784,0.007740,0.001836,0.000142,0.001414,0.002225,0.003178,0.005672,0.035338,0.001813,0.004394,0.853289,0.000935,0.002778,0.012708,0.000816,0.883981,0.107177,0.008842,12319816
986,12319816_001,13,Общее впечатление о банке,-1,"В общем, работать с данным банком не рекомендую.",0.007456,0.009121,0.006549,0.007348,0.003115,0.000337,0.002462,0.007418,0.003664,0.010418,0.098511,0.004851,0.773587,0.010846,0.002380,0.036408,0.008335,0.007194,0.877237,0.097528,0.025234,12319816
987,12319816_002,13,Общее впечатление о банке,-1,Будете разочарованы.,0.051812,0.011087,0.007297,0.010302,0.010152,0.003958,0.007511,0.005077,0.021458,0.017913,0.397107,0.030989,0.328808,0.052269,0.004421,0.003032,0.034259,0.002547,0.559206,0.306871,0.133923,12319816
988,12319816_003,16,Сравнение с конкурентами,-1,Тиньков работает лучше.,0.061036,0.012094,0.019737,0.009121,0.005794,0.007726,0.013477,0.008465,0.048092,0.024556,0.612093,0.011077,0.079147,0.036843,0.002509,0.027204,0.018731,0.002298,0.696436,0.187235,0.116329,12319816
2064,12319816_000,14,"Кэшбэк, промокоды, бонусы, акции, приведи друга",-2,"Получила 20.05.25 дебетовую карту газпромбанка в офисе,т.к. банк проводит акцию по выплате кэшбэка в размере 35% за коммунальные услуги.Работники банка меня уверили,что с 01.03.25г. остаток на карте должен был оставаться 10 000,0 руб., а не 30 000 руб. В итоге кэшбэка нет, еще и сняли 399 руб. за неизвестные привилегии.",0.058922,0.004014,0.002784,0.007740,0.001836,0.000142,0.001414,0.002225,0.003178,0.005672,0.035338,0.001813,0.004394,0.853289,0.000935,0.002778,0.012708,0.000816,0.883981,0.107177,0.008842,12319816
2065,12319816_001,13,Общее впечатление о банке,-2,"В общем, работать с данным банком не рекомендую.",0.007456,0.009121,0.006549,0.007348,0.003115,0.000337,0.002462,0.007418,0.003

#### sentences

#### eval val_df

In [ ]:
reviews_ds.head()

,id,title,review_text,overall_rating
0,11458385,не возвращают денежные средства за оформление карт друзьями по реферальным ссылкам,"в марте этого года жена оформляла кредитную карту по моей реферальной ссылке. были выполнены все условия для получения вознаграждения - 2500 рублей с карт были соверешены покупки более 3000 рублей. обратился по этому вопросу в чат техподдержки. долгое ожидание оператора, оператор ничем не пытается помочь в решении вопроса, сразу выходит из чата, не дав задать вопрос. быть может, сам банк и неплохой, но ужасные отношения к клиенту, нет попыток даже помочь решить ситуацию. в этом месяце также были оформлены 2 дебетовые карты по моей реф. ссылке, но в техподдержке снова пояснили, что по моей ссылке не было оформлено карт. принял решение заблокировать все карты и счета данного банка",2.0
1,12210503,24.02.2025,"я столкнулась с ситуацией, когда система газпромбанка некорректно распознала категорию покупки и не начислила кэшбэк в должном объеме. я приобрела билеты в кино через специализированный сайт кинопоиска, который является партнером кинотеатров, но кэшбэк за покупку так и не поступил. 24 февраля 2025 года я столкнулась с ситуацией, когда система газпромбанка некорректно распознала категорию покупки и не начислила кэшбэк в должном объеме. я приобрел билеты в кино через специализированный сайт кинопоиска, который является партнером кинотеатров, но кэшбэк за покупку так и не пришел спасибо за подробности! теперь я могу составить отзыв, который точно соответствует требованиям и будет зачтен.",5.0
2,11578417,"не надо сюда! берегите нервы, деньги и время!","являюсь премиальным корпоративным клиентом, поэтому знаю, о чем пишу. пользовалась большинством продуктов банка и с каждым!!! были проблемы. ипотека оформляется с 10 утра до 5 вечера, вся цепочка сделки сидит ждёт, недоумевая, почему нельзя подготовить документы заранее. премиальный менеджер заявлен только номинально. большая текучка кадров. чат в мобильном приложении обслуживают люди с низким уровнем квалификации, медленно и грубо. максимум, что могут сделать, - составить запрос в тех. поддержку. тех. поддержка проблемы не решает. каждую неделю присылает смс, что решение вопроса продлено ещё на месяц!. например, не приходят заказанные выписки и справки на указанный адрес эл. почты. решают больше года. по кредитной карте расчет льготного периода идет календарными месяцами если вы открыли карту 20 ноября, льготный период начался ещё 1 ноября, т.е. минус 20 дней. если погасили задолженность 2 декабря, новый льготный период начнется только 1 января. к слову, в других банках льготный период начинает исчисляться сразу. по вкладам повышенная процентная ставка действует только в первые пару месяцев. по потреб.кредитам навязана невыгодная страховка. кешбек гораздо менее выгодный, чем в других банках даже для премиальных клиентов. карты работают через раз выдали мир, но оказалась кобрендовой с мастеркард, поэтому как мир не определяется, за границей не работает. оформили цифровую jcb, потом сами признались, что не заработала ни у кого. оформили union pay. цифровая не работает, пластиковая при оплате тоже не прошла. точкой невозврата стало исчезновение счета с деньгами из мобильного приложения при перекидывании с карты в мобильном приложении. счет перестал быть виден везде, кроме как у операторов по телефону, баланс средств уменьшился. только личные связи менеджера с техническими специалистами позволили восстановить информацию за пару дней, а не за 30, как по умолчанию. вынуждена была перенести повседневные операции в другой банк, хотя, конечно, было бы удобно пользоваться банком, в который приходит зарплата.",1.0
3,11891226,"прошло 83 дня, а приветственного сертификата на 1000 от озон так и нет","20.08.2024 оформила дебетовую карту мир газпромбанка 1528. настоял на этом муж, мол оформи и приветственный сертификат получишь на 1000 рублей от ozon и акция 35 на всё. так и поступила. карту заказала, акцию подключила, деньги начала тратить. согласно условиям акции,

In [ ]:
# 2  11314037  [1, 22, 24, 21, 30]
reviews_ds[reviews_ds['id']==11314037]

,id,title,review_text,overall_rating
14145,11314037,обман сотрудников,"здравствуйте. хочу оставить отзыв на качество обслуживания, как в офисе, так и чате мобильного приложения. получила карту, с горем пополам, в офисе, хотя была курьерская доставка. в итоге, просидела час и соизволила ко мне спуститься специалист. кое-как выдали карту. сказала отключить услуги платные сразу, на что оператор ответил, что делать это нужно через приложение. а я не могла её активировать и сделать что либо , т к при выдаче карты указали номер некорректно. спустя день или два, я пришла вновь в офис. сотруднику было ясно сказано, чтобы отключили все платные услуги страховки и смс уведомления тоже. в декабре 2023 года чудным образом у меня списывают деньги за услугу смс-инфо, хотя я, уверена что она отключена. с горем пополам регистрируют обращение на возврат и, конечно же, приходит отказ. операторы в чате просто идиоты! отвечают по полдня. переписку предыдущую не читают! требуют регистрации жалобы на сотрудников офиса, ноль реакции. хочу узнать как работает офис 6го числа и могу ли закрыть карту, отвечают по пол дня и говорят идите в офис. совсем идиоты?! что за дебилы в вас там работают? они знаю русский язык? а слова лояльность, клиентоориентирванность, до претензионная работа - им знакомы?? позорище, а не банк! дай бог я закрою карту в скопом времени, не исключено, конечно, что будут новые приключения с вашими сотрудниками...",1.0


In [ ]:
val_df[val_df['sentence_id'].str.contains('11313905')]

,sentence_id,cat_id,categories,sentiment,sentence_text,processed_correctly
349,11313905_2,1,[Дебетовые карты],1,по карте была проведена непонятная операция 30,True
373,11313905_4,21,[Работа колл-центра],1,много часов пыталась получить ответ от поддерж...,True
381,11313905_7,24,[Общее впечатление о банке],1,обслуживание - дно,True
403,11313905_5,21,[Работа колл-центра],1,"финальный ответ просто убил к сожалению, у нас...",True
416,11313905_8,30,[прочее],1,"жаль, что из-за санкций практически нет альтер...",True
590,11313905_6,1,[Дебетовые карты],1,"карту буду закрывать, категорически не советую...",True
596,11313905_3,30,[прочее],1,"12, которую в выписке не нашла",True
600,11313905_0,24,[Общее впечатление о банке],1,хуже обслуживания никогда не встречала,True


In [ ]:
'''а если по кредитной?
11314037 	обман сотрудников 	здравствуйте. хочу оставить отзыв на качество обслуживания, как в офисе, так и чате мобильного приложения. получила карту, с горем пополам, в офисе, хотя была курьерская доставка. в итоге, просидела час и соизволила ко мне спуститься специалист. кое-как выдали карту. сказала отключить услуги платные сразу, на что оператор ответил, что делать это нужно через приложение. а я не могла её активировать и сделать что либо , т к при выдаче карты указали номер некорректно. спустя день или два, я пришла вновь в офис. сотруднику было ясно сказано, чтобы отключили все платные услуги страховки и смс уведомления тоже. в декабре 2023 года чудным образом у меня списывают деньги за услугу смс-инфо, хотя я, уверена что она отключена. с горем пополам регистрируют обращение на возврат и, конечно же, приходит отказ. операторы в чате просто идиоты! отвечают по полдня. переписку предыдущую не читают! требуют регистрации жалобы на сотрудников офиса, ноль реакции. хочу узнать как работает офис 6го числа и могу ли закрыть карту, отвечают по пол дня и говорят идите в офис. совсем идиоты?! что за дебилы в вас там работают? они знаю русский язык? а слова лояльность, клиентоориентирванность, до претензионная работа - им знакомы?? позорище, а не банк! дай бог я закрою карту в скопом времени, не исключено, конечно, что будут новые приключения с вашими сотрудниками...
моя разметка выделила
[1, 22, 24, 21, 30]
{1: 'Дебетовые карты',
 22: 'Работа отделений',
 24: 'Общее впечатление о банке',
30 прочее - должно отбрасываться

'''

In [ ]:
CATEGORIES_D

{1: 'Дебетовые карты',
 2: 'Кредитные карты',
 3: 'Вклады',
 4: 'Накопительные счета',
 5: 'Потребительские кредиты',
 6: 'Автокредиты',
 7: 'Ипотека',
 8: 'Кредиты под залог недвижимости',
 9: 'Рефинансирование кредитов',
 10: 'Инвестиции',
 11: 'Страхование',
 12: 'Обмен валюты',
 13: 'Денежные переводы',
 14: 'Мобильный банк и приложение',
 15: 'Интернет-банк',
 16: 'Премиум-обслуживание',
 17: 'Газпром Бонус',
 18: 'ГПБ Мобайл',
 19: 'Банковские сейфы',
 20: 'Кредитная история и рейтинг',
 21: 'Работа колл-центра',
 22: 'Работа отделений',
 23: 'Безопасность',
 24: 'Общее впечатление о банке',
 25: 'Брокерский счет / ИИС',
 26: 'Сберегательные счета в металлах',
 27: 'Пенсионные продукты',
 28: 'Реструктуризация кредита',
 29: 'Кэшбэк, промокоды, бонусы, акции',
 30: 'прочее'}